In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType

orders_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("customer_id", IntegerType()),
    StructField("quantity", IntegerType()),
    StructField("total", DoubleType()),
    StructField("books", ArrayType(
        StructType([
            StructField("book_id", IntegerType()),
            StructField("qty", IntegerType())
        ])
    ))
])

bronze_orders_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", "/Volumes/demoworkspace_new/default/my_volume/ecommerce/schemas/orders")
    .schema(orders_schema)
    .load("/Volumes/demoworkspace_new/default/my_volume/ecommerce/orders_raw/"))

(bronze_orders_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/demoworkspace_new/default/my_volume/ecommerce/checkpoints/bronze_orders")
    .trigger(availableNow=True)
    .table("bronze_orders"))

In [0]:
customers_df = spark.read.json("/Volumes/demoworkspace_new/default/my_volume/ecommerce/customers.json")
customers_df.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")

In [0]:
from pyspark.sql.functions import to_timestamp, col

silver_orders_df = (spark.readStream.table("bronze_orders")
    .withColumn("timestamp", to_timestamp("timestamp"))
    .dropDuplicates(["order_id"])
    .filter(col("total") > 0))

(silver_orders_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/demoworkspace_new/default/my_volume/ecommerce/checkpoints/silver_orders")
    .trigger(availableNow=True)
    .table("silver_orders"))

In [0]:
silver_customers_df = (spark.read.table("bronze_customers")
    .select(
        "customer_id",
        "email",
        col("profile.first_name").alias("first_name"),
        col("profile.last_name").alias("last_name"),
        col("profile.gender").alias("gender"),
        col("profile.address.city").alias("city"),
        col("profile.address.country").alias("country"),
        to_timestamp("updated_at").alias("updated_at")
    ))

silver_customers_df.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

In [0]:
from pyspark.sql.functions import window, sum as _sum

gold_hourly_revenue_df = (spark.readStream.table("silver_orders")
    .withWatermark("timestamp", "10 minutes")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_sum("total").alias("revenue")))

(gold_hourly_revenue_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "/Volumes/demoworkspace_new/default/my_volume/ecommerce/checkpoints/gold_hourly_revenue")
    .trigger(availableNow=True)
    .table("gold_hourly_revenue"))

In [0]:
customer_summary_df = (spark.read.table("silver_orders")
    .join(spark.read.table("silver_customers"), on="customer_id", how="inner")
    .groupBy("customer_id", "first_name", "last_name", "city")
    .agg(
        _sum("total").alias("total_spent"),
        _sum("quantity").alias("total_items_ordered")
    ))

customer_summary_df.write.format("delta").mode("overwrite").saveAsTable("gold_customer_summary")

In [0]:
%sql
SELECT * FROM gold_hourly_revenue;